#

The idea is to start by making a route from one point to another

# Functions

In [11]:
import requests
import json
import folium

In [ ]:
'''
    Takes a list of waypoints (longitude, latitude)
    and returns routing information from OSRM, including distance and duration for each route. 
'''
def get_route_osrm(waypoints, alternatives=2):
    # Build coordinate string from waypoints list
    coords = ";".join([f"{lon},{lat}" for lon, lat in waypoints])
    
    url = f"https://router.project-osrm.org/route/v1/driving/{coords}"

    params = {
        "overview": "full",
        "geometries": "geojson",
        "steps": "true",
        "alternatives": alternatives
    }

    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()
    data = response.json()

    if data.get("code") != "Ok":
        raise ValueError(f"Routing failed: {data}")

    return data

'''
    Use folioum to plot the routes returned by OSRM on an interactive map. Each route will be displayed as a blue line.
'''
def plot_osrm_routes(data, out_file="osrm_routes.html"):
    if not data.get("routes"):
        print("No routes found to plot.")
        return

    # Get the first route's geometry for centering the map
    first_route = data["routes"][0]
    first_coords = first_route["geometry"]["coordinates"]
    center_lat = sum(lat for lon, lat in first_coords) / len(first_coords)
    center_lon = sum(lon for lon, lat in first_coords) / len(first_coords)

    m = folium.Map(location=[center_lat, center_lon], zoom_start=5)

    for route in data["routes"]:
        coords = route["geometry"]["coordinates"]
        folium.PolyLine(locations=[(lat, lon) for lon, lat in coords], color="blue", weight=5).add_to(m)

    # Save the map to an HTML file
    m.save(out_file)
    return m

def plot_osrm_routes_with_disaster(data, disaster_location, disaster_radius, out_file="osrm_routes_with_disaster.html"):
    m = plot_osrm_routes(data)

    # Add a circle to represent the disaster area
    folium.Circle(
        location=[disaster_location[1], disaster_location[0]],  # lat, lon
        radius=disaster_radius,
        color='red',
        fill=True,
        fill_color='red',
        fill_opacity=0.5,
        popup='Disaster Area'
    ).add_to(m)

    m.save(out_file)

    return m


'''
    Check if a route crosses a disaster area defined by a center point and radius.
'''
def route_crosses_disaster_area(route, disaster_location, disaster_radius):
    for lon, lat in route["geometry"]["coordinates"]:
        distance = ((lon - disaster_location[0]) ** 2 + (lat - disaster_location[1]) ** 2) ** 0.5
        if distance <= disaster_radius:
            return True
    return False


# Generate and check route

We make a route and a distaster, then we check if the route crosses the disaster area

In [13]:
waypoints = [
    (2.3522, 48.8566),  # Paris
    (4.9041, 52.3676),  # Amsterdam
    (10.7522, 59.9139)  # Oslo
]

disatster_location = (10.0, 55.5)  # Odense, Denmark
disatster_radius = 100000  # 100 km

data = get_route_osrm(waypoints=waypoints)

print(f"Number of routes returned with waypoints: {len(data['routes'])}")
for i, route in enumerate(data["routes"], start=1):
    print(f"Route {i}: {route['distance']} meters, {route['duration']} seconds") 

map = plot_osrm_routes(data)

for i, route in enumerate(data["routes"], start=1):
    if route_crosses_disaster_area(route, disatster_location, disatster_radius):
        print(f"Route {i} crosses the disaster area.")
    else:
        print(f"Route {i} does NOT cross the disaster area.")

map = plot_osrm_routes_with_disaster(data, disatster_location, disatster_radius)




Number of routes returned with waypoints: 1
Route 1: 2024773.4 meters, 81344.5 seconds
Route 1 crosses the disaster area.
